In [21]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("F1_Preguntas")
    .enableHiveSupport()
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark.sql("USE f1_dw")

DataFrame[]

In [22]:
# Pregunta 1
q1 = spark.sql("""
SELECT nombre, apellido, victorias, pos
FROM (
    SELECT d.givenName  AS nombre,
           d.familyName AS apellido,
           COUNT(*)     AS victorias,
           RANK() OVER (ORDER BY COUNT(*) DESC) AS pos
    FROM fact_results f
    JOIN dim_driver d ON f.driverId = d.driverId
    WHERE f.position = 1
    GROUP BY d.givenName, d.familyName
) t
WHERE pos <= 10
ORDER BY pos, apellido, nombre
""")

q1.show(truncate=False)

+---------+-----------+---------+---+
|nombre   |apellido   |victorias|pos|
+---------+-----------+---------+---+
|Lewis    |Hamilton   |105      |1  |
|Max      |Verstappen |71       |2  |
|Michael  |Schumacher |56       |3  |
|Sebastian|Vettel     |53       |4  |
|Fernando |Alonso     |32       |5  |
|Nico     |Rosberg    |23       |6  |
|Kimi     |Räikkönen  |21       |7  |
|Jenson   |Button     |15       |8  |
|Rubens   |Barrichello|11       |9  |
|Felipe   |Massa      |11       |9  |
|Lando    |Norris     |11       |9  |
+---------+-----------+---------+---+



In [23]:
# Pregunta 2
q2 = spark.sql("""
SELECT c.circuitName AS circuito,
       c.country     AS pais,
       c.latitude    AS lat,
       c.longitude   AS lon,
       COUNT(DISTINCT f.raceId)                                    AS carreras,
       COUNT(*)                                                    AS largaron,
       SUM(CASE WHEN f.positionText = 'R' THEN 1 ELSE 0 END)       AS dnf,
       ROUND(100.0 * SUM(CASE WHEN f.positionText = 'R' THEN 1 ELSE 0 END)
             / COUNT(*), 2)                                        AS tasa_dnf_pct
FROM fact_results f
JOIN dim_race    r ON f.raceId    = r.raceId
JOIN dim_circuit c ON r.circuitId = c.circuitId
WHERE f.positionText <> 'W'          -- excluye a los que no largaron (withdrew / DNS)
GROUP BY c.circuitId, c.circuitName, c.country, c.latitude, c.longitude
HAVING COUNT(DISTINCT f.raceId) >= 5
ORDER BY tasa_dnf_pct DESC
LIMIT 10
""")

q2.show(truncate=False)

+------------------------------+---------+--------+--------+--------+--------+---+------------+
|circuito                      |pais     |lat     |lon     |carreras|largaron|dnf|tasa_dnf_pct|
+------------------------------+---------+--------+--------+--------+--------+---+------------+
|Indianapolis Motor Speedway   |USA      |39.795  |-86.2347|8       |154     |58 |37.66       |
|Albert Park Grand Prix Circuit|Australia|-37.8497|144.968 |24      |497     |154|30.99       |
|Hockenheimring                |Germany  |49.3278 |8.56583 |14      |299     |79 |26.42       |
|Circuit de Monaco             |Monaco   |43.7347 |7.42056 |25      |521     |137|26.30       |
|Nürburgring                   |Germany  |50.3356 |6.9475  |12      |255     |66 |25.88       |
|Sepang International Circuit  |Malaysia |2.76083 |101.738 |18      |387     |100|25.84       |
|Circuit Gilles Villeneuve     |Canada   |45.5    |-73.5228|23      |488     |120|24.59       |
|Autodromo Enzo e Dino Ferrari |Italy   

In [24]:
# Pregunta 3
q3 = spark.sql("""
SELECT nombre, apellido, carreras_validas, ganancia_promedio, pos
FROM (
    SELECT d.givenName  AS nombre,
           d.familyName AS apellido,
           COUNT(*)     AS carreras_validas,
           ROUND(AVG(f.grid - f.position), 2) AS ganancia_promedio,
           RANK() OVER (ORDER BY AVG(f.grid - f.position) DESC) AS pos
    FROM fact_results f
    JOIN dim_driver d ON f.driverId = d.driverId
    WHERE f.positionText RLIKE '^[0-9]+$'   -- solo clasificados (excluye R, D, W)
      AND f.grid > 0                        -- excluye largadas desde boxes
    GROUP BY d.givenName, d.familyName
    HAVING COUNT(*) >= 20 -- mínima cantidad de carreras para considerar piloto
) t
WHERE pos <= 10
ORDER BY pos, apellido, nombre;
""");
q3.show(truncate=False)

+----------+----------+----------------+-----------------+---+
|nombre    |apellido  |carreras_validas|ganancia_promedio|pos|
+----------+----------+----------------+-----------------+---+
|Jos       |Verstappen|29              |6.48             |1  |
|Jean      |Alesi     |21              |5.67             |2  |
|Mika      |Salo      |23              |5.26             |3  |
|Alexander |Wurz      |25              |4.52             |4  |
|Tiago     |Monteiro  |29              |4.38             |5  |
|Christijan|Albers    |28              |4.14             |6  |
|Pedro     |de la Rosa|51              |3.67             |7  |
|Charles   |Pic       |30              |3.53             |8  |
|Oliver    |Bearman   |24              |3.33             |9  |
|Eddie     |Irvine    |25              |3.32             |10 |
+----------+----------+----------------+-----------------+---+



In [25]:
# Pregunta 4
q4 = spark.sql("""
WITH podios_por_combo AS (
    SELECT d.givenName  AS nombre,
           d.familyName AS apellido,
           c.name       AS constructor,
           COUNT(*)     AS podios,
           RANK() OVER (ORDER BY COUNT(*) DESC) AS pos
    FROM fact_results f
    JOIN dim_driver      d ON f.driverId      = d.driverId
    JOIN dim_constructor c ON f.constructorId = c.constructorId
    WHERE f.position <= 3
    GROUP BY d.givenName, d.familyName, c.name
)
SELECT nombre, apellido, constructor, podios, pos
FROM podios_por_combo
WHERE pos <= 10
ORDER BY pos, apellido, nombre
""");
q4.show(truncate=False)

+---------+-----------+-----------+------+---+
|nombre   |apellido   |constructor|podios|pos|
+---------+-----------+-----------+------+---+
|Lewis    |Hamilton   |Mercedes   |153   |1  |
|Max      |Verstappen |Red Bull   |127   |2  |
|Michael  |Schumacher |Ferrari    |83    |3  |
|Sebastian|Vettel     |Red Bull   |65    |4  |
|Valtteri |Bottas     |Mercedes   |58    |5  |
|Rubens   |Barrichello|Ferrari    |55    |6  |
|Nico     |Rosberg    |Mercedes   |55    |6  |
|Sebastian|Vettel     |Ferrari    |55    |6  |
|Kimi     |Räikkönen  |Ferrari    |52    |9  |
|Charles  |Leclerc    |Ferrari    |50    |10 |
+---------+-----------+-----------+------+---+



In [26]:
# Pregunta 5
q5 = spark.sql("""
    WITH dnf_por_escuderia AS (
        SELECT
            r.constructorId,
            SUM(CASE WHEN r.positionText = 'R' THEN 1 ELSE 0 END)          AS dnf_count,
            SUM(CASE WHEN r.positionText <> 'W' THEN 1 ELSE 0 END)         AS starts,
            ROUND(
                SUM(CASE WHEN r.positionText = 'R' THEN 1 ELSE 0 END) * 100.0
                / SUM(CASE WHEN r.positionText <> 'W' THEN 1 ELSE 0 END), 2
            )                                                              AS dnf_rate_pct
        FROM f1_dw.fact_results r
        JOIN f1_dw.dim_race ra ON r.raceId = ra.raceId
        WHERE ra.season BETWEEN 2016 AND 2025
        GROUP BY r.constructorId
    ),
    posicion_final AS (
        SELECT
            constructorId,
            COUNT(*)                    AS seasons,
            ROUND(AVG(position), 2)     AS avg_final_position
        FROM f1_dw.fact_constructor_standings
        WHERE season BETWEEN 2016 AND 2025
        GROUP BY constructorId
    )
    SELECT
        c.name                AS constructor,
        d.dnf_count,
        d.starts,
        d.dnf_rate_pct,
        p.seasons,
        p.avg_final_position
    FROM dnf_por_escuderia d
    JOIN posicion_final p ON d.constructorId = p.constructorId
    JOIN f1_dw.dim_constructor c ON d.constructorId = c.constructorId
    ORDER BY d.dnf_count DESC
""")
q5.show(truncate=False)

+--------------+---------+------+------------+-------+------------------+
|constructor   |dnf_count|starts|dnf_rate_pct|seasons|avg_final_position|
+--------------+---------+------+------------+-------+------------------+
|Toro Rosso    |37       |166   |22.29       |4      |7.25              |
|Renault       |44       |199   |22.11       |5      |5.8               |
|Manor Marussia|8        |42    |19.05       |1      |11.0              |
|Sauber        |40       |219   |18.26       |5      |9.4               |
|Williams      |70       |425   |16.47       |10     |7.9               |
|Haas F1 Team  |65       |424   |15.33       |10     |8.3               |
|Racing Point  |10       |75    |13.33       |2      |5.5               |
|AlphaTauri    |21       |164   |12.80       |4      |7.5               |
|Red Bull      |54       |427   |12.65       |10     |2.3               |
|Alfa Romeo    |26       |208   |12.50       |5      |8.0               |
|RB F1 Team    |12       |96    |12.50